# Lab Assignment 4: Using APIs in Python
## DS 6001: Practice and Application of Data Science
### Robert Clay Harris

### Instructions
Please answer the following questions as completely as possible using text, code, and the results of code as needed. Format your answers in a Jupyter notebook. To receive full credit, make sure you address every part of the problem, and make sure your document is formatted in a clean and professional way.

In this lab, you will work with the public API provided by [genius.com](https://genius.com/), a website that calls itself "the world’s biggest collection of song lyrics and musical knowledge." You will need to read the API documentation carefully, acquire an access key, and use it without sharing it to pull data from this API into Python. You will also practice using a library made specifically to wrap around `requests` to make calling from the Genius API easier.

## Problem 0
Import the following libraries:

In [1]:
import numpy as np
import pandas as pd
import requests
import json
import os
import dotenv
import sys
sys.tracebacklimit = 0 # turn off the error tracebacks

## Problem 1
The Genius API documentation is here: https://docs.genius.com/#/getting-started-h1. Read through the documentation carefully. Although the Genius API is free and public, it still requires users an access key to use the API. In this case, Genius provides users with three codes: a client ID, a client ID secret, and a client access token. Use the documentation to find a way to obtain these codes for yourself. Write a paragraph that describes all of the steps you needed to take (but DO NOT list your access codes in this paragraph).

Some hints and cautions: 

1. Before you can use the API, you will need a regular, free user account with Genius. Sign-up here: https://genius.com/signup_or_login

2. Genius's API is built to support third-party app development, not data scientists. The language is entirely geared toward app development. Under "Authentication" there are instructions to third-party developers for guiding their own users in getting access keys. That's not relevant to getting access for yourself. It's not hard to get an access key, but the guidance here is not very clear. Be patient and read everything in the Authentication section carefully.

3. When you arrive at the page that allows you to register for API access keys, the language is still geared toward app development. You will be prompted to name your app and provide the URLs associated with the app. It doesn't much matter what you name your app, and I just used the Collab main page (https://collab.its.virginia.edu/portal?containerLogin=true) for the URLs.

4. When you get your codes, copy them in a text file. In problem 2 you will copy these codes over again to a `.env` file. [4 points]

To obtain access to the Genius API, I first created a free user account on Genius.com though gmail. After logging in, I navigated to the Genius API documentation and reviewed the Authentication section. Since Genius’s API is designed for third-party apps, I registered an application by visiting the Genius Developer Portal and clicking “Create an API Client.” In the application form, I provided a name for my app and used the URL to UVA Collab since an actual app URL was not necessary for personal API access. Once submitted, Genius generated three credentials: a Client ID, Client Secret, and Client Access Token. I copied these codes and stored them in a text file for later use in setting up a .env file.

## Problem 2
Create a `.env` file for this project. Open it, copy your access codes into it, and save it. Then use Python code to load the environmental variables contained in the `.env` file, and create variables that contain each of the three codes. (You can print these variables to make sure it worked, but do not allow your access keys to display in your notebook). [4 points]

In [2]:
dotenv.load_dotenv()

# Retrieve the Genius API credentials
genius_id = os.getenv("genius_id")
genius_secret = os.getenv("genius_secret")
genius_token = os.getenv("genius_token")

assert genius_id is not None, "genius_id not found"
assert genius_secret is not None, "genius_secret not found"
assert genius_token is not None, "genius_token not found"

## Problem 3
The root for all Genius APIs is https://api.genius.com. Find the endpoint for the Search API. (You will have to click the "Authorize with Genius" button in the upper-right corner if you haven't already done so). Use the `requests` library to issue a search for Bob Dylan. Genius's API is organized in a way that every individual artist has his or her own API endpoint. Display a portion of the JSON output that displays the API endpoint path for the data on Bob Dylan. 

Hint: to authenticate, specify your access token (not your client ID or client secret) as the `access_token` parameter. You will have to dig around the JSON output to find the artist ID, but it is listed under `primary_artist` several branches down the JSON tree. [4 points]

In [3]:
# Set up the root URL and endpoint for the Genius API Search
root = "https://api.genius.com"
endpoint = "/search"
url = root + endpoint

# Define the parameters for the search request:
params = {
    'q': 'Bob Dylan',
    'access_token': genius_token
}

# Issue the GET request
r = requests.get(url, params=params)
r

<Response [200]>

In [4]:
# Load the JSON response
myjson = r.json()

# we locate the 'primary_artist' object and retrieve its 'api_path'.
artist_api_path = myjson['response']['hits'][0]['result']['primary_artist']['api_path']

result = {"Bob Dylan Artist API Endpoint": artist_api_path}
result

{'Bob Dylan Artist API Endpoint': '/artists/181'}

## Problem 4
Add `/songs` to the end of the the endpoint path you found in problem 3 and use this path to request the 20 most popular Bob Dylan songs. Organize these data in a `pandas` data frame. [4 points]

In [5]:
songs_url = root + artist_api_path + "/songs"

# Set up parameters: sort by popularity, limit to 20 songs, include access token
params = {
    "id": 181,
    "sort": "popularity",
    "per_page": 20,
    "access_token": genius_token
}

# Issue the GET request to fetch the songs data
response = requests.get(songs_url, params=params)

songs_json = response.json()

songs_df = pd.json_normalize(songs_json["response"]["songs"])
songs_df.head(3)

,annotation_count,api_path,artist_names,full_title,header_image_thumbnail_url,header_image_url,id,lyrics_owner_id,lyrics_state,path,...,stats.pageviews,primary_artist.api_path,primary_artist.header_image_url,primary_artist.id,primary_artist.image_url,primary_artist.is_meme_verified,primary_artist.is_verified,primary_artist.name,primary_artist.url,stats.concurrents
0,15,/songs/96286,USA For Africa,We Are the World by USA For Africa,https://images.genius.com/d328397b28953bd84465...,https://images.genius.com/d328397b28953bd84465...,96286,4733728,complete,/Usa-for-africa-we-are-the-world-lyrics,...,699943,/artists/370890,https://images.genius.com/3fb2d9f68c911b547339...,370890,https://images.genius.com/3fb2d9f68c911b547339...,False,False,USA For Africa,https://genius.com/artists/Usa-for-africa,NaN
1,12,/songs/79424,Bob Dylan,Blowin' in the Wind by Bob Dylan,https://images.genius.com/84e1705bc64495197216...,https://images.genius.com/84e1705bc64495197216...,79424,73267,complete,/Bob-dylan-blowin-in-the-wind-lyrics,...,652577,/artists/181,https://images.genius.com/e87fb11dd7f33cc7fd1a...,181,https://images.genius.com/a94817af6c91a49c07fa...,False,False,Bob Dylan,https://genius.com/artists/Bob-dylan,NaN
2,1,/songs/68146,Adele,Make You Feel My Love by Adele,https://images.genius.com/f248cc85ccaab603ce88...,https://images.genius.com/f248cc85ccaab603ce88...,68146,82481,complete,/Adele-make-you-feel-my-love-lyrics,...,643290,/artists/2300,https://images.genius.com/87aa5d8c32965a10e0e7...,2300,https://images.genius.com/8a23ab928ccfce13accf...,False,False,Adele,https://genius.com/artists/Adele,2.0


## Problem 5
Install and import the `lyricsgenius` library in Python, which is a wrapper around `requests` that works specifically with the Genius API. . Follow the guide on the GitHub repository for this library (https://github.com/johnwmillr/LyricsGenius) for instructions on using the library. Use the `lyricsgenius` library to download and display the lyrics to "Tangled Up in Blue" by Bob Dylan. [4 points]

In [6]:
import lyricsgenius

genius = lyricsgenius.Genius(genius_token, verbose=False, remove_section_headers=True)

song = genius.search_song("Tangled Up in Blue", "Bob Dylan")

with open("tangled_up_in_blue.txt", "w") as f:
    f.write(song.lyrics)

# Split the lyrics into lines
lines = song.lyrics.split('\n')

# Print the first 10 lines
print("\n".join(lines[:10]))

103 ContributorsTangled Up in Blue Lyrics
Early one morning the sun was shining
I was laying in bed
Wondering if she'd changed at all
If her hair was still red
Her folks they said our lives together
Sure was going to be rough
They never did like Mama's homemade dress
Papa's bankbook wasn't big enough
And I was standing on the side of the road
